In [1]:
import numpy

In [7]:
import os
import re
import random
import tarfile
import requests
from io import BytesIO

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

In [6]:
from sentence_transformers import SentenceTransformer


In [8]:
def get_IMDB():
    """
    Download and extract the Stanford ACL IMDB movie reviews dataset.
    Returns a DataFrame with two columns:
      - 'text': the review text
      - 'label': 1 for positive reviews, 0 for negative reviews
    This function extracts the training data from the archive.
    """
    data_url = 'http://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
    
    # Download the tar.gz archive from Stanford.
    response = requests.get(data_url, stream=True)
    gz_file = response.content  # Raw bytes from the response.
    
    # Open the tar file from the downloaded bytes.
    tar = tarfile.open(fileobj=BytesIO(gz_file), mode="r:gz")
    
    reviews = []
    labels = []
    
    # Iterate over all members of the tar file.
    for member in tar.getmembers():
        if member.isfile():
            # We're interested in the training set files only.
            if "aclImdb/train/pos/" in member.name:
                file_obj = tar.extractfile(member)
                if file_obj is not None:
                    # Read file, decode and strip any extraneous whitespace.
                    review = file_obj.read().decode("utf-8", errors="ignore").strip()
                    reviews.append(review)
                    labels.append(1)
            elif "aclImdb/train/neg/" in member.name:
                file_obj = tar.extractfile(member)
                if file_obj is not None:
                    review = file_obj.read().decode("utf-8", errors="ignore").strip()
                    reviews.append(review)
                    labels.append(0)
                    
    # Build and return a DataFrame.
    df = pd.DataFrame({"text": reviews, "label": labels})
    return df

# Download dataset (training data).
print("Downloading IMDB dataset...")
imdb_df = get_IMDB()
print("Number of training samples in full dataset:", imdb_df.shape[0])
print(imdb_df.head(2))

Number of training samples in full dataset: 25000
                                                text  label
0  I rented I AM CURIOUS-YELLOW from my video sto...      0
1  "I Am Curious: Yellow" is a risible and preten...      0


In [29]:
X = imdb_df['text'].tolist()
y = imdb_df['label'].tolist()

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Subsampled Training set size:", len(X_train))
print("Subsampled Test set size:", len(X_test))

Subsampled Training set size: 20000
Subsampled Test set size: 5000


In [31]:
vectorizer = CountVectorizer(lowercase=True, stop_words='english')
X_train_counts = vectorizer.fit_transform(X_train)
X_test_counts  = vectorizer.transform(X_test)

In [32]:
doc_clf = LogisticRegression(max_iter=200)
doc_clf.fit(X_train_counts, y_train)

LogisticRegression(max_iter=200)

In [33]:
y_pred = doc_clf.predict(X_test_counts)
baseline_accuracy = accuracy_score(y_test, y_pred)
baseline_auc = roc_auc_score(y_test, doc_clf.predict_proba(X_test_counts)[:, 1])
print("\nBaseline Document Classifier Accuracy: {:.2f}".format(baseline_accuracy))
print("Baseline Document Classifier AUC: {:.2f}".format(baseline_auc))


Baseline Document Classifier Accuracy: 0.88
Baseline Document Classifier AUC: 0.95


In [50]:
coef = doc_clf.coef_[0]
vocab = np.array(vectorizer.get_feature_names_out())
num_candidates = 30  # Adjust as needed.
top_indices = np.argsort(np.abs(coef))[::-1][:num_candidates]
candidate_words = vocab[top_indices]
print("\nCandidate Words from Initial Classifier:")
print(candidate_words)


Candidate Words from Initial Classifier:
['waste' 'disappointment' 'worst' 'poorly' 'awful' 'lacks' 'wonderfully'
 'alright' 'baldwin' 'excellent' 'mess' 'disappointing' 'incredible'
 'boring' 'avoid' 'badly' 'annoying' 'superb' 'rare' 'weak' 'horrible'
 'worse' 'funniest' 'perfectly' 'refreshing' 'hoping' 'perfect' 'amazing'
 'highly' 'surprisingly']


In [51]:
model_name = 'all-MiniLM-L6-v2'
embedder = SentenceTransformer(model_name)
print("\nPrecomputing embeddings for training reviews...")
X_train_embeddings = embedder.encode(X_train, convert_to_numpy=True)


Precomputing embeddings for training reviews...


In [52]:
def extract_context(sentence, target_word, window=5):
    """
    Extract the context surrounding the target word within the given window.
    Returns a list of context strings for all occurrences of target_word.
    """
    words = sentence.split()
    indices = [i for i, word in enumerate(words) if word.lower() == target_word.lower()]
    contexts = []
    for idx in indices:
        start = max(0, idx - window)
        end = min(len(words), idx + window + 1)
        context = words[start:idx] + words[idx+1:end]
        contexts.append(" ".join(context))
    return contexts

In [53]:
def perform_matching(word, sentences, labels, precomputed_embeddings):
    """
    For every review that contains the target 'word', compute:
      - Average cosine similarity between the batch-encoded context embeddings and the candidate reviews' embeddings.
      - A simplified Average Treatment Effect (ATE): review label minus average candidate label.
      - Total count of matching instances.
    
    Uses precomputed embeddings for efficiency.
    """
    similarities = []
    treatment_effects = []
    count_matches = 0

    for i, sentence in enumerate(sentences):
        if re.search(r'\b' + re.escape(word.lower()) + r'\b', sentence.lower()):
            contexts = extract_context(sentence, word)
            if not contexts:
                continue
            # Batch encode contexts.
            context_embeddings = embedder.encode(contexts, convert_to_numpy=True)
            # Identify indices of candidate reviews that do NOT contain the word.
            candidate_indices = [j for j, s in enumerate(sentences) if word.lower() not in s.lower()]
            if not candidate_indices:
                continue
            candidate_embeddings = precomputed_embeddings[candidate_indices]
            for emb in context_embeddings:
                cos_sim = cosine_similarity([emb], candidate_embeddings)
                max_sim = np.max(cos_sim)
                similarities.append(max_sim)
                # Simplified treatment effect: difference between review label and average candidate label.
                candidate_labels = [labels[j] for j in candidate_indices]
                avg_candidate_label = np.mean(candidate_labels)
                treatment_effects.append(labels[i] - avg_candidate_label)
                count_matches += 1

    if count_matches == 0:
        return 0.0, 0.0, 0
    return np.mean(similarities), np.mean(treatment_effects), count_matches

In [54]:

features = []
print("\nComputing matching features for candidate words...")
for word in candidate_words:
    avg_sim, avg_ate, match_count = perform_matching(word, X_train, y_train, X_train_embeddings)
    # Also use the word's coefficient from our document classifier.
    word_coef = coef[list(vocab).index(word)]
    features.append({
        'word': word,
        'avg_similarity': avg_sim,
        'avg_ate': avg_ate,
        'match_count': match_count,
        'word_coef': word_coef
    })

df_features = pd.DataFrame(features)
print("\nCandidate Word Features:")
print(df_features)


Computing matching features for candidate words...

Candidate Word Features:
              word  avg_similarity   avg_ate  match_count  word_coef
0            waste        0.453611 -0.468275         1113  -2.206077
1   disappointment        0.464713 -0.173781          151  -2.088253
2            worst        0.510146 -0.447528         1955  -1.889778
3           poorly        0.519320 -0.429054          513  -1.743684
4            awful        0.510818 -0.398205          746  -1.724195
5            lacks        0.502882 -0.227718          286  -1.510592
6      wonderfully        0.517888  0.411868          238   1.405013
7          alright        0.511069 -0.402795           60  -1.338111
8          baldwin        0.471961 -0.455193           43  -1.305513
9        excellent        0.517190  0.333594         1290   1.300435
10            mess        0.459996 -0.328334          266  -1.299247
11   disappointing        0.494467 -0.307572          202  -1.297437
12      incredible       

In [55]:
len(df_features)

30

In [56]:
# Modified simulated labeling, mark the candidate word with the highest absolute coefficient as spurious.
# (Assuming candidate_words is ordered by decreasing absolute coefficient.)
word_label_mapping = {word: (1 if i == 0 else 0) for i, word in enumerate(candidate_words)}

# Update the DataFrame with the simulated labels.
df_features['spurious_label'] = df_features['word'].apply(lambda w: word_label_mapping.get(w, 0))
print("\nFeatures with Simulated Human Labels:")
print(df_features)



Features with Simulated Human Labels:
              word  avg_similarity   avg_ate  match_count  word_coef  \
0            waste        0.453611 -0.468275         1113  -2.206077   
1   disappointment        0.464713 -0.173781          151  -2.088253   
2            worst        0.510146 -0.447528         1955  -1.889778   
3           poorly        0.519320 -0.429054          513  -1.743684   
4            awful        0.510818 -0.398205          746  -1.724195   
5            lacks        0.502882 -0.227718          286  -1.510592   
6      wonderfully        0.517888  0.411868          238   1.405013   
7          alright        0.511069 -0.402795           60  -1.338111   
8          baldwin        0.471961 -0.455193           43  -1.305513   
9        excellent        0.517190  0.333594         1290   1.300435   
10            mess        0.459996 -0.328334          266  -1.299247   
11   disappointing        0.494467 -0.307572          202  -1.297437   
12      incredible       

In [57]:
word_feature_columns = ['avg_similarity', 'avg_ate', 'match_count', 'word_coef']
X_word = df_features[word_feature_columns].values
y_word = df_features['spurious_label'].values

In [58]:
word_clf = LogisticRegression()
word_clf.fit(X_word, y_word)



LogisticRegression()

In [59]:
df_features['predicted_spurious_prob'] = word_clf.predict_proba(X_word)[:, 1]
print("\nWord Classifier Predictions:")
print(df_features[['word', 'predicted_spurious_prob']])


Word Classifier Predictions:
              word  predicted_spurious_prob
0            waste                 0.136073
1   disappointment                 0.025941
2            worst                 0.349365
3           poorly                 0.038676
4            awful                 0.055814
5            lacks                 0.022075
6      wonderfully                 0.002505
7          alright                 0.013651
8          baldwin                 0.013069
9        excellent                 0.016461
10            mess                 0.018724
11   disappointing                 0.016715
12      incredible                 0.003340
13          boring                 0.049147
14           avoid                 0.025629
15           badly                 0.023735
16        annoying                 0.029229
17          superb                 0.003562
18            rare                 0.003343
19            weak                 0.024364
20        horrible                 0.032787
21

In [66]:
spurious_threshold = 0.3
words_to_remove = df_features[df_features['predicted_spurious_prob'] > spurious_threshold]['word'].tolist()
print("\nWords predicted as spurious and to be removed:", words_to_remove)


Words predicted as spurious and to be removed: ['worst']


In [67]:
def custom_preprocessor(text):
    pattern = r'\b(?:' + '|'.join(words_to_remove) + r')\b'
    return re.sub(pattern, '', text, flags=re.IGNORECASE)

In [68]:
vectorizer_fs = CountVectorizer(lowercase=True, stop_words='english', preprocessor=custom_preprocessor)
X_train_fs = vectorizer_fs.fit_transform(X_train)
X_test_fs  = vectorizer_fs.transform(X_test)

In [69]:
doc_clf_fs = LogisticRegression(max_iter=200)
doc_clf_fs.fit(X_train_fs, y_train)


LogisticRegression(max_iter=200)

In [70]:
y_pred_fs = doc_clf_fs.predict(X_test_fs)
fs_accuracy = accuracy_score(y_test, y_pred_fs)
fs_auc = roc_auc_score(y_test, doc_clf_fs.predict_proba(X_test_fs)[:, 1])
print("\nDocument Classifier after Feature Selection:")
print("  Accuracy: {:.2f}".format(fs_accuracy))
print("  AUC: {:.2f}".format(fs_auc))


Document Classifier after Feature Selection:
  Accuracy: 0.88
  AUC: 0.95


In [71]:
def contains_spurious(text, spurious_words):
    """
    Check if the given text contains any of the spurious words (case-insensitive).
    """
    return any(re.search(r'\b' + re.escape(word) + r'\b', text, re.IGNORECASE) for word in spurious_words)


In [72]:
spurious_markers = words_to_remove


In [73]:
# Split the original test samples into two groups:
#  - Minority group: test samples that contain at least one spurious word.
#  - Majority group: test samples that do not.
indices_minority = [i for i, text in enumerate(X_test) if contains_spurious(text, spurious_markers)]
indices_majority = [i for i in range(len(X_test)) if i not in indices_minority]

X_test_fs_array = X_test_fs  # Already transformed features

In [74]:
if indices_minority:
    y_pred_minority = doc_clf_fs.predict(X_test_fs_array[indices_minority])
    acc_minority = accuracy_score(np.array(y_test)[indices_minority], y_pred_minority)
    auc_minority = roc_auc_score(np.array(y_test)[indices_minority], doc_clf_fs.predict_proba(X_test_fs_array[indices_minority])[:, 1])
else:
    acc_minority = None
    auc_minority = None

if indices_majority:
    y_pred_majority = doc_clf_fs.predict(X_test_fs_array[indices_majority])
    acc_majority = accuracy_score(np.array(y_test)[indices_majority], y_pred_majority)
    auc_majority = roc_auc_score(np.array(y_test)[indices_majority], doc_clf_fs.predict_proba(X_test_fs_array[indices_majority])[:, 1])
else:
    acc_majority = None
    auc_majority = None


In [75]:
print("\nSubgroup Analysis:")
print("Minority Group (contains spurious markers):")
print("  Number of samples:", len(indices_minority))
print("  Accuracy: {}".format(acc_minority))
print("  AUC: {}".format(auc_minority))


Subgroup Analysis:
Minority Group (contains spurious markers):
  Number of samples: 456
  Accuracy: 0.9100877192982456
  AUC: 0.9479470648116728


In [76]:
print("\nMajority Group (does NOT contain spurious markers):")
print("  Number of samples:", len(indices_majority))
print("  Accuracy: {}".format(acc_majority))
print("  AUC: {}".format(auc_majority))


Majority Group (does NOT contain spurious markers):
  Number of samples: 4544
  Accuracy: 0.8756602112676056
  AUC: 0.9428846266300217


In [77]:
def augment_training_data(texts, spurious_words):
    """
    For each training review containing any spurious word, create a counterfactual example
    by removing the spurious words.
    Returns a list of augmented reviews.
    """
    augmented_texts = []
    for text in texts:
        if any(re.search(r'\b' + re.escape(word) + r'\b', text, re.IGNORECASE) for word in spurious_words):
            # Remove all spurious words.
            augmented = re.sub(r'\b(?:' + '|'.join(spurious_words) + r')\b', '', text, flags=re.IGNORECASE)
            # Clean up extra spaces.
            augmented = re.sub(' +', ' ', augmented).strip()
            augmented_texts.append(augmented)
    return augmented_texts

In [78]:
print("\nGenerating counterfactual augmented training data...")
# Use the spurious words predicted to be removed as markers.
augmented_X_train = augment_training_data(X_train, words_to_remove)
print("Number of augmented examples created:", len(augmented_X_train))


Generating counterfactual augmented training data...
Number of augmented examples created: 1813


In [79]:
X_train_combined = X_train + augmented_X_train
# For labels, assume augmented texts maintain the same labels;
# here we repeat the corresponding labels (for simplicity, we can pair them one-to-one based on occurrence).
# (Note: In practice, we may need to align labels more carefully.)
y_train_combined = y_train + y_train[:len(augmented_X_train)]


In [80]:
# Train a new document classifier on the augmented training set.
vectorizer_aug = CountVectorizer(lowercase=True, stop_words='english')
X_train_combined_counts = vectorizer_aug.fit_transform(X_train_combined)
X_test_aug_counts = vectorizer_aug.transform(X_test)


In [81]:

doc_clf_aug = LogisticRegression(max_iter=200)
doc_clf_aug.fit(X_train_combined_counts, y_train_combined)


/Users/rahul/Documents/study/purdue/Sem 2/ECE AI/project/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=200)

In [82]:
y_pred_aug = doc_clf_aug.predict(X_test_aug_counts)
aug_accuracy = accuracy_score(y_test, y_pred_aug)
aug_auc = roc_auc_score(y_test, doc_clf_aug.predict_proba(X_test_aug_counts)[:, 1])
print("\nDocument Classifier with Counterfactual Augmentation:")
print("  Accuracy: {:.2f}".format(aug_accuracy))
print("  AUC: {:.2f}".format(aug_auc))


Document Classifier with Counterfactual Augmentation:
  Accuracy: 0.85
  AUC: 0.92


In [83]:
def contains_spurious(text, spurious_words):
    """Return True if text contains any of the spurious words (case-insensitive)."""
    return any(re.search(r'\b' + re.escape(word) + r'\b', text, re.IGNORECASE) for word in spurious_words)


In [84]:
spurious_markers = words_to_remove


In [85]:
indices_minority = [i for i, text in enumerate(X_test) if contains_spurious(text, spurious_markers)]
indices_majority = [i for i in range(len(X_test)) if i not in indices_minority]



In [86]:
X_test_aug = X_test_aug_counts  # using the augmented CountVectorizer (same features as vectorizer_aug)


In [87]:
if indices_minority:
    y_pred_minority = doc_clf_aug.predict(X_test_aug[indices_minority])
    acc_minority = accuracy_score(np.array(y_test)[indices_minority], y_pred_minority)
    auc_minority = roc_auc_score(np.array(y_test)[indices_minority], doc_clf_aug.predict_proba(X_test_aug[indices_minority])[:, 1])
else:
    acc_minority = auc_minority = None

In [88]:
if indices_majority:
    y_pred_majority = doc_clf_aug.predict(X_test_aug[indices_majority])
    acc_majority = accuracy_score(np.array(y_test)[indices_majority], y_pred_majority)
    auc_majority = roc_auc_score(np.array(y_test)[indices_majority], doc_clf_aug.predict_proba(X_test_aug[indices_majority])[:, 1])
else:
    acc_majority = auc_majority = None

In [89]:
print("\nExtended Subgroup Analysis (Augmented Classifier):")
print("Minority Group (contains spurious markers):")
print("  Samples:", len(indices_minority), " Accuracy:", acc_minority, " AUC:", auc_minority)
print("Majority Group (does not contain spurious markers):")
print("  Samples:", len(indices_majority), " Accuracy:", acc_majority, " AUC:", auc_majority)


Extended Subgroup Analysis (Augmented Classifier):
Minority Group (contains spurious markers):
  Samples: 456  Accuracy: 0.9517543859649122  AUC: 0.9084492704445197
Majority Group (does not contain spurious markers):
  Samples: 4544  Accuracy: 0.8389084507042254  AUC: 0.9135200670526089
